<a href="https://colab.research.google.com/github/yonipick-subair/MiniProject_1/blob/main/Q1_neural_spikes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1: Modeling and Analyzing Neural Spike Patterns

**Mini-Project 1 · Neuroscience Data Analysis with Python**

We simulate three neurons over a 5-second recording (regular, irregular and bursting firing), analyze how their spike trains differ, validate the simulation, and examine variability across realizations.

| Section | Owner |
|---|---|
| A. Constructing the models | Yonatan |
| B. Investigating the firing patterns | Junia |
| C. Testing our assumptions | Yonatan |
| D. Investigating new realizations | Yonatan |

## Setup

All randomness goes through a single `numpy.random.Generator` created from a fixed seed, so the notebook gives identical results every time it runs.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42            # global seed for reproducibility
DURATION = 5.0       # recording length in seconds (required by the assignment)

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110


def save_fig(fig, name):
    """Save a figure to figures/ with a consistent naming scheme (q1_<name>.png)."""
    path = FIG_DIR / f"q1_{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path

## A. Constructing the models
*Owner: Yonatan*

### General approach

Every neuron is described by its **sequence of inter-spike intervals (ISIs)**. A spike train is built by drawing ISIs and taking their cumulative sum:

$$t_k = \sum_{i=1}^{k} \text{ISI}_i$$

Since every ISI is strictly positive, the cumulative sum is **strictly increasing by construction**. This is how we guarantee that spike times are valid and chronologically ordered: we never need to sort. ISIs are drawn in blocks until the running time passes the recording length (5 s), and spikes at or after 5 s are discarded.

The three neurons differ only in **how the ISIs are drawn**. All three are tuned to the **same mean firing rate (10 Hz)**, so that any difference between them comes from the temporal structure of the spikes and not from how many spikes there are. This lets section B test directly whether mean rate alone can distinguish them.

### Modeling assumptions

1. **Regular neuron:** ISIs are nearly constant. Each ISI is the nominal interval (1/rate = 100 ms) plus small Gaussian noise (SD = 2% of the interval), representing small biological fluctuations. Without the noise the model would be fully deterministic.
2. **Irregular neuron:** a **homogeneous Poisson process**. ISIs are independent and exponentially distributed with mean 1/rate. This is the standard model of a neuron firing at random with a constant probability per unit time.
3. **Bursting neuron:** spikes come in bursts of a fixed number of spikes with a short, fixed ISI inside the burst (5 ms). Between bursts there is a quiet period made of a **minimum gap** (50 ms, so bursts never merge) plus an exponentially distributed random part. The random part's mean is chosen so that bursts occur on average at 2 Hz, giving 2 × 5 = 10 spikes/s.
4. **Recording:** spikes are recorded in the interval [0, 5) s. The recording starts at t = 0, so the first spike occurs one ISI after the start.
5. **Not modeled** (simplifications): absolute refractory period for the regular and Poisson neurons, rate changes over time, and interactions between neurons.

### Parameters

| Neuron | Parameter | Value | Meaning | Deterministic / stochastic |
|---|---|---|---|---|
| All | `DURATION` | 5 s | Length of the recording | Deterministic |
| All | `SEED` | 42 | Seed of the random generator | Fixes the stochastic parts |
| Regular | `rate` | 10 Hz | Nominal firing rate → nominal ISI = 100 ms | Deterministic |
| Regular | `jitter_frac` | 0.02 | SD of the ISI noise, as a fraction of the nominal ISI | Controls the stochastic part |
| Irregular | `rate` | 10 Hz | Mean firing rate → mean ISI = 100 ms | ISIs are stochastic (exponential) |
| Bursting | `burst_rate` | 2 Hz | Mean number of bursts per second | Burst timing is stochastic |
| Bursting | `spikes_per_burst` | 5 | Spikes in each burst | Deterministic |
| Bursting | `intra_isi` | 5 ms | ISI inside a burst | Deterministic |
| Bursting | `min_gap` | 50 ms | Minimum quiet interval between bursts | Deterministic |

In [4]:
# ---------- Model parameters (all values documented in the table above) ----------
PARAMS = {
    "regular":   {"rate": 10.0, "jitter_frac": 0.02},
    "irregular": {"rate": 10.0},
    "bursting":  {"burst_rate": 2.0, "spikes_per_burst": 5, "intra_isi": 0.005, "min_gap": 0.050},
}

In [5]:
def _spike_times_from_isis(draw_isis, duration):
    """Turn a generator of ISIs into spike times in [0, duration).

    draw_isis: function with no arguments that returns a 1-D array of positive ISIs (one block).
    Blocks are drawn until the cumulative time passes `duration`; spikes at or after
    `duration` are discarded. Because every ISI is > 0, the result is strictly increasing.
    """
    if duration <= 0:
        return np.array([])
    blocks, t = [], 0.0
    while t < duration:
        isis = np.asarray(draw_isis(), dtype=float)
        if np.any(isis <= 0):
            raise ValueError("All ISIs must be strictly positive.")
        times = t + np.cumsum(isis)
        blocks.append(times)
        t = times[-1]
    spikes = np.concatenate(blocks)
    return spikes[spikes < duration]


def simulate_regular(rate, duration, rng, jitter_frac=0.02):
    """Regular neuron: ISI = 1/rate plus small Gaussian noise (SD = jitter_frac * 1/rate)."""
    if rate <= 0:
        return np.array([])
    nominal_isi = 1.0 / rate
    block = int(duration * rate) + 10

    def draw():
        isis = nominal_isi * (1 + jitter_frac * rng.standard_normal(block))
        # Noise this small never makes an ISI negative, but we guard against it explicitly.
        return np.clip(isis, 1e-4, None)

    return _spike_times_from_isis(draw, duration)


def simulate_irregular(rate, duration, rng):
    """Irregular neuron: homogeneous Poisson process, ISIs ~ Exponential(mean = 1/rate)."""
    if rate <= 0:
        return np.array([])
    block = int(duration * rate) + 10
    return _spike_times_from_isis(lambda: rng.exponential(1.0 / rate, size=block), duration)


def simulate_bursting(burst_rate, spikes_per_burst, intra_isi, min_gap, duration, rng):
    """Bursting neuron: bursts of `spikes_per_burst` spikes separated by `intra_isi`.

    The quiet interval between the last spike of one burst and the first spike of the next is
    min_gap + Exponential(mean_extra), where mean_extra is chosen so that one full burst cycle
    lasts 1/burst_rate seconds on average.
    """
    if burst_rate <= 0 or spikes_per_burst <= 0:
        return np.array([])
    cycle = 1.0 / burst_rate
    mean_extra = cycle - (spikes_per_burst - 1) * intra_isi - min_gap
    if mean_extra <= 0:
        raise ValueError("Burst parameters are inconsistent: bursts would not fit in one cycle.")

    def draw():
        # One cycle = one quiet interval followed by (spikes_per_burst - 1) intra-burst ISIs.
        quiet = min_gap + rng.exponential(mean_extra)
        return np.concatenate([[quiet], np.full(spikes_per_burst - 1, intra_isi)])

    return _spike_times_from_isis(draw, duration)

In [6]:
# ---------- One realization of each neuron, with the global seed ----------
rng = np.random.default_rng(SEED)

spikes = {
    "regular":   simulate_regular(duration=DURATION, rng=rng, **PARAMS["regular"]),
    "irregular": simulate_irregular(duration=DURATION, rng=rng, **PARAMS["irregular"]),
    "bursting":  simulate_bursting(duration=DURATION, rng=rng, **PARAMS["bursting"]),
}

# Quick sanity check that each train is valid (full validation is in section C).
for name, s in spikes.items():
    assert s.ndim == 1 and np.all(np.diff(s) > 0) and s.min() >= 0 and s.max() < DURATION, name
    print(f"{name:<10} {len(s):3d} spikes  ({len(s) / DURATION:.1f} Hz)   first spikes [s]: {np.round(s[:5], 3)}")

regular     49 spikes  (9.8 Hz)   first spikes [s]: [0.101 0.199 0.3   0.402 0.498]
irregular   47 spikes  (9.4 Hz)   first spikes [s]: [0.151 0.682 0.892 1.057 1.161]
bursting    55 spikes  (11.0 Hz)   first spikes [s]: [0.103 0.108 0.113 0.118 0.123]


## B. Investigating the firing patterns
*Owner: Junia*

Design a visual and numerical analysis comparing the three neurons. Our analysis answers:

1. How does each neuron distribute its spikes over time?
2. What can a raster plot reveal?
3. What information is contained in the inter-spike intervals (ISI)?
4. How do the ISI distributions differ?
5. Can mean firing rate alone distinguish the three neurons?
6. Which additional statistics help characterize their firing patterns?

For every figure and statistic, explain **why** it was chosen.

### B.1 Raster plot

In [7]:
# TODO (Junia): raster plot of the three neurons. Save with save_fig(fig, "raster").

### B.2 Inter-spike interval distributions

In [8]:
# TODO (Junia): ISI histograms (np.diff on spike times), one panel per neuron, shared x-axis.
# Save with save_fig(fig, "isi_hist").

### B.3 Summary statistics

In [9]:
# TODO (Junia): table with spike count, mean rate (Hz), mean ISI, ISI std and CV for each neuron.

### B.4 Answers to the questions above
<!-- One short paragraph per question, referring to the figures and the table. -->

## C. Testing our assumptions
*Owner: Yonatan*

A simulation can produce plausible-looking data while implementing the wrong model.
We design at least three checks that verify the model we **intended**, not only that the code runs:

| # | Type | What it checks | Expected result |
|---|---|---|---|
| 1 | Temporal behavior | | |
| 2 | Statistical property | | |
| 3 | Edge case | | |

In [10]:
# TODO (Yonatan): Check 1 – temporal behavior (e.g. all ISIs positive, sorted, inside [0, DURATION)).

In [11]:
# TODO (Yonatan): Check 2 – statistical property (e.g. Poisson CV ≈ 1, regular CV ≈ 0, across many seeds).

In [12]:
# TODO (Yonatan): Check 3 – edge case (e.g. very short duration, zero rate, same seed → identical output).

### What the checks show
<!-- For each check: what was tested, what we expected, what we got. -->

## D. Investigating new realizations
*Owner: Yonatan*

- Generate additional realizations of the stochastic models.
- How much do spike patterns vary between realizations?
- Can two realizations of the same neuron model look quite different?
- Which properties remain relatively stable?

In [13]:
# TODO (Yonatan): run each model over many seeds, collect rate and CV, and plot their distributions.
# Save with save_fig(fig, "realizations").

### Distinguishing the three firing patterns from observations
<!-- How would we tell the three patterns apart from recorded data, without knowing how they were generated? -->

## Model description and scientific interpretation
<!-- Required for submission: a brief description of the models and assumptions, and a short scientific interpretation. -->

### Models and assumptions

### Interpretation